# NFR classification — revision experiments

**How to run:** Runtime → Change runtime type → **T4 GPU**, then **Runtime → Run all**. That is all.

* Upload `nfr_five_provenance.csv` when asked (first run only).
* The last cell runs every experiment as a separate job (TensorFlow models and PyTorch fine-tuning never share a process,
  and each job releases its memory when it ends). A job that crashes is retried automatically from its checkpoints.
* Expected time on a T4: about 3–4 hours. Keep the tab open. If Colab disconnects completely, just **Run all** again:
  finished jobs are skipped (set `USE_DRIVE = True` in the last cell to keep progress in Google Drive).
* At the end `nfr_revision_results.zip` downloads automatically — send that file back.

| Job group | Reviewer comment |
|---|---|
| Protocol A: original 10 splits, all models + tuned baselines + RoBERTa + class-weighted hybrid + block attribution | R1.1, R1.2, R1.6, R1.7, R1.9, R1.10, R2.2, R2.3 |
| Protocol B: de-duplicated corpus, duplicate-aware repeated 5-fold CV | R1.1, R1.3 |
| Protocol C: leave-one-project-out | R1.3, R1.8, R2.4 |
| End-to-end timing on GPU and CPU | R1.5, R2.1 |
| Summary: corrected resampled t-tests, Holm, project bootstrap | all |


### 1. Core experiment code (written to a file; nothing runs here)

In [ ]:
%%writefile /content/nfr_core.py
# NFR revision experiments -- core script. Run through the driver notebook (one job per process).
# # NFR classification — revision experiments (single consistent run)
#
# This notebook re-executes **all** experiments requested in the major revision in one environment:
#
# | Block | What it does | Reviewer comment |
# |---|---|---|
# | Protocol A | Original 10 stratified 80/20 splits (seeds 34–43), all 9 original models + tuned baselines + fine-tuned RoBERTa + class-weighted hybrid | R1.1, R1.2, R1.6, R1.7, R1.10, R2.3 |
# | Protocol B | Exact duplicates removed, near-duplicate groups never split, repeated (2×) stratified group 5-fold CV | R1.1, R1.3 |
# | Protocol C | Leave-one-project-out over the 15 PROMISE-NFR projects that are traceable in the dataset | R1.3, R1.8, R2.4 |
# | Attribution | Block-level (TF-IDF vs DistilBERT) exact Shapley values, block ablation, expected-gradients attributions over all 5,768 inputs | R1.8, R1.9, R2.2 |
# | Timing | End-to-end training and inference on GPU **and** CPU, embedding extraction included | R1.5, R2.1 |
# | Summary | Tables, corrected resampled t-tests (Nadeau–Bengio), Holm correction, project-cluster bootstrap | all |
#
# **How to run (Google Colab, T4 GPU):**
# 1. Runtime → Change runtime type → T4 GPU.
# 2. Upload `nfr_five_provenance.csv` (supplied with the revision) when prompted, or put it in `MyDrive/nfr_revision/`.
# 3. Run all cells. Each individual model fit is check-pointed to Google Drive, so if Colab disconnects just re-run all cells and it resumes where it stopped.
# 4. At the end, download `nfr_revision_results.zip` (also saved in Drive) and send it back.
#
# Expected total time on a T4: roughly 3–4 hours.

PHASE = 1   # 1 = TensorFlow models, 2 = PyTorch fine-tuning (set per job by the driver)
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
LOCAL_TEST = os.environ.get("NFR_LOCAL_TEST", "0") == "1"   # only used to smoke-test this notebook outside Colab
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "memory_profiler"], check=False)
    # triton (a PyTorch add-on for torch.compile, not used here) loads a second copy of LLVM that clashes with
    # TensorFlow's XLA and silently kills the kernel when the first transformer is fine-tuned. Remove it before torch loads.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "triton"], check=False)
os.environ["TORCHDYNAMO_DISABLE"] = "1"   # torch.compile is not used
PHASE = int(os.environ.get("NFR_PHASE", PHASE))
STAGE = os.environ.get("NFR_STAGE", "run")              # run | timing | summary  (set by the driver notebook)
JOB_PROTOCOL = os.environ.get("NFR_PROTOCOL", "")       # A | B | C
JOB_MODELS = [m for m in os.environ.get("NFR_MODELS", "").split(",") if m]
if PHASE == 2:   # keep TensorFlow out of this session entirely
    os.environ["USE_TF"] = "0"; os.environ["TRANSFORMERS_NO_TF"] = "1"

import json, time, gc, re, random, math, platform, shutil, glob, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

USE_DRIVE = False   # True = save progress to Google Drive (survives disconnects); False = Colab's own disk
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/nfr_revision"
elif IN_COLAB:
    BASE = "/content/nfr_revision"
else:
    BASE = os.environ.get("NFR_BASE", os.path.abspath("./nfr_revision"))
os.makedirs(BASE, exist_ok=True)
OUT_DIR = os.path.join(BASE, "results")
os.makedirs(OUT_DIR, exist_ok=True)

DATA_PATH = os.path.join(BASE, "nfr_five_provenance.csv")
if not os.path.exists(DATA_PATH):
    if IN_COLAB:
        from google.colab import files
        print("Upload nfr_five_provenance.csv")
        up = files.upload()
        name = list(up.keys())[0]
        shutil.move(name, DATA_PATH)
    elif os.path.exists("nfr_five_provenance.csv"):
        shutil.copy("nfr_five_provenance.csv", DATA_PATH)
print("Data:", DATA_PATH, "| Output:", OUT_DIR)

# ------------------------------------------------------------------ switches
RUN_PROTOCOL_A = True
RUN_PROTOCOL_B = True
RUN_PROTOCOL_C = True
RUN_ATTRIBUTION = True
RUN_TIMING = True

SEEDS_A = list(range(34, 44))      # identical to the manuscript
REPEATS_B = [34, 35]               # 2 x 5-fold = 10 paired estimates
FOLDS_B = 5
INNER_VAL = 0.15
TEST_SIZE = 0.20
MAX_FEATURES = 5000
KERAS_EPOCHS, KERAS_BATCH = 12, 32
TR_MAX_EPOCHS, TR_BATCH, TR_MAX_LEN, TR_WD, TR_WARMUP, TR_PATIENCE = 12, 16, 192, 0.01, 0.1, 3
TR_LR_GRID = [1e-5, 2e-5, 3e-5, 5e-5]     # selected once on the inner validation split of seed 34
LINEAR_C_GRID = [0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100]
KERAS_GRID = [dict(lr=1e-3, drop=1.0), dict(lr=1e-3, drop=0.5), dict(lr=3e-4, drop=1.0), dict(lr=3e-4, drop=0.5)]
DISTILBERT = "distilbert-base-uncased"
ROBERTA = "roberta-base"

if LOCAL_TEST:   # tiny settings so the whole pipeline can be smoke-tested on a CPU without internet
    SEEDS_A, REPEATS_B, KERAS_EPOCHS, TR_MAX_EPOCHS = [34, 35], [34], 2, 1
    TR_LR_GRID, LINEAR_C_GRID = [2e-5, 5e-5], [0.1, 1, 10]
    KERAS_GRID = KERAS_GRID[:2]

CATS = ['Maintainability', 'Operability', 'Performance', 'Security', 'Usability']
K = len(CATS)

# ## Data, preprocessing and split generators

import nltk
os.environ.setdefault("NLTK_ALLOW_PROXIED_URLOPEN", "1")
for r in ["stopwords", "wordnet", "omw-1.4"]:
    nltk.download(r, quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split, StratifiedGroupKFold

REPLACE_BY_SPACE = re.compile(r'[/(){}|@,;]')
STOPWORDS = set(stopwords.words("english")) - {"not", "no", "nor"}
LEM = WordNetLemmatizer()

def clean_text(text):
    text = REPLACE_BY_SPACE.sub(" ", str(text).lower())
    text = re.sub(r"\s+", " ", text).strip()
    return " ".join(LEM.lemmatize(t) for t in text.split() if t not in STOPWORDS)

df = pd.read_csv(DATA_PATH)
assert len(df) == 914 and (df.row_id.values == np.arange(914)).all()
df["sentence"] = df["sentence"].fillna("")
df["text_clean"] = df["sentence"].map(clean_text)
df["y"] = df["class_name"].map({c: i for i, c in enumerate(CATS)}).astype(int)
Y_ALL = df["y"].values
SENT = df["sentence"].tolist()
CLEAN = df["text_clean"].tolist()
print(df.class_name.value_counts().to_dict())
print("dedup rows:", int(df.keep_dedup.sum()), "| traceable projects:", int(df.promise_exp_project.nunique()))

def _inner_val_group(pool_idx, seed):
    """Group-aware inner validation split (~1/7 of the pool) so that near-duplicates never straddle train/val."""
    sg = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=seed)
    tr, va = next(sg.split(pool_idx, Y_ALL[pool_idx], groups=df.near_dup_group.values[pool_idx]))
    return pool_idx[tr], pool_idx[va]

def splits_A():
    """Original protocol: identical calls to the manuscript's code (seeds 34-43)."""
    for s in SEEDS_A:
        outer_tr, te = train_test_split(np.arange(len(df)), test_size=TEST_SIZE, random_state=s, stratify=Y_ALL)
        tr, va = train_test_split(outer_tr, test_size=INNER_VAL, random_state=s, stratify=Y_ALL[outer_tr])
        yield dict(fold=f"seed{s}", seed=s, outer_train=outer_tr, train=tr, val=va, test=te)

def splits_B():
    """Deduplicated data, near-duplicate groups kept together, repeated stratified group 5-fold CV."""
    dd = np.where(df.keep_dedup.values)[0]
    for r in REPEATS_B:
        sg = StratifiedGroupKFold(n_splits=FOLDS_B, shuffle=True, random_state=r)
        for k, (tr_, te_) in enumerate(sg.split(dd, Y_ALL[dd], groups=df.near_dup_group.values[dd])):
            outer_tr, te = dd[tr_], dd[te_]
            tr, va = _inner_val_group(outer_tr, r * 100 + k)
            yield dict(fold=f"r{r}_k{k}", seed=r * 100 + k, outer_train=outer_tr, train=tr, val=va, test=te)

def splits_C():
    """Leave-one-project-out over the PROMISE-NFR projects that are traceable in the dataset.
    Rows without project metadata are only ever used for training; near-duplicates of test rows are removed from training."""
    dd = df[df.keep_dedup]
    projects = sorted(dd.promise_exp_project.dropna().unique())
    if LOCAL_TEST:
        projects = projects[:3]
    for p in projects:
        te = dd.index[dd.promise_exp_project == p].values
        test_groups = set(df.near_dup_group.values[te])
        pool = dd.index[(dd.promise_exp_project != p) & (~dd.near_dup_group.isin(test_groups))].values
        tr, va = _inner_val_group(pool, int(p))
        yield dict(fold=f"P{int(p):02d}", seed=1000 + int(p), outer_train=pool, train=tr, val=va, test=te)

# sanity: no leakage in B / C
for gen in (splits_B, splits_C):
    for sp in gen():
        g_tr = set(df.near_dup_group.values[sp["outer_train"]]); g_te = set(df.near_dup_group.values[sp["test"]])
        assert not (g_tr & g_te), "near-duplicate leakage"
print("Fold counts: A=%d B=%d C=%d" % (len(list(splits_A())), len(list(splits_B())), len(list(splits_C()))))

# ## Models (architectures identical to the manuscript) and feature builders

USE_TF = (PHASE == 1)
if USE_TF:
    import tensorflow as tf      # import TensorFlow before PyTorch (avoids an LLVM symbol clash in some builds)
    for _g in tf.config.list_physical_devices("GPU"):
        tf.config.experimental.set_memory_growth(_g, True)
    from tensorflow.keras import layers, models as kmodels, callbacks
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if PHASE == 2: assert "tensorflow" not in sys.modules, "TensorFlow is loaded: restart the session before running PHASE 2"
print("PHASE", PHASE, "| torch", torch.__version__, "| tf", tf.__version__ if USE_TF else "not loaded", "| device", DEVICE,
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if USE_TF: tf.keras.utils.set_random_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def macro_f1(y, p): return f1_score(y, p, average="macro", labels=list(range(K)), zero_division=0)

# ---------------------------------------------------------------- transformer loading
def load_tok_model(name, kind):
    """kind = 'encoder' (frozen feature extractor) or 'cls' (sequence classifier)."""
    if LOCAL_TEST:
        return _tiny_local_model(name, kind)
    tok = AutoTokenizer.from_pretrained(name)
    if kind == "encoder":
        return tok, AutoModel.from_pretrained(name)
    return tok, AutoModelForSequenceClassification.from_pretrained(name, num_labels=K)

def _tiny_local_model(name, kind):  # offline smoke test only
    from tokenizers import Tokenizer as HFTok, models as hm, pre_tokenizers, trainers
    from transformers import PreTrainedTokenizerFast, DistilBertConfig, RobertaConfig
    t = HFTok(hm.WordLevel(unk_token="[UNK]")); t.pre_tokenizer = pre_tokenizers.Whitespace()
    t.train_from_iterator(SENT, trainers.WordLevelTrainer(special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]"]))
    tok = PreTrainedTokenizerFast(tokenizer_object=t, pad_token="[PAD]", unk_token="[UNK]")
    if "roberta" in name:
        cfg = RobertaConfig(vocab_size=t.get_vocab_size() + 2, hidden_size=32, num_hidden_layers=1, num_attention_heads=2,
                            intermediate_size=64, max_position_embeddings=260, num_labels=K, pad_token_id=0)
    else:
        cfg = DistilBertConfig(vocab_size=t.get_vocab_size(), dim=32, n_layers=1, n_heads=2, hidden_dim=64, num_labels=K, pad_token_id=0)
    m = AutoModel.from_config(cfg) if kind == "encoder" else AutoModelForSequenceClassification.from_config(cfg)
    return tok, m

# ---------------------------------------------------------------- frozen DistilBERT sentence embeddings
def embed_texts(texts, tok, enc, device, batch=32):
    enc.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(texts), batch):
            b = tok(texts[i:i + batch], return_tensors="pt", padding=True, truncation=True, max_length=192).to(device)
            h = enc(**{k: v for k, v in b.items() if k in ("input_ids", "attention_mask")}).last_hidden_state
            m = b["attention_mask"].unsqueeze(-1).float()
            out.append(((h * m).sum(1) / m.sum(1).clamp(min=1e-9)).cpu().numpy())
    return normalize(np.vstack(out).astype(np.float32), norm="l2")

EMB_PATH = os.path.join(BASE, "distilbert_embeddings.npy" if not LOCAL_TEST else "tiny_embeddings.npy")
if os.path.exists(EMB_PATH):
    BERT_EMB = np.load(EMB_PATH)
else:
    _tok, _enc = load_tok_model(DISTILBERT, "encoder"); _enc.to(DEVICE)
    BERT_EMB = embed_texts(SENT, _tok, _enc, DEVICE); np.save(EMB_PATH, BERT_EMB)
    del _enc; gc.collect()
print("DistilBERT embeddings:", BERT_EMB.shape)

# ---------------------------------------------------------------- Keras architectures (manuscript, Section 3.3)
def build_dense(input_dim, deep=True, lr=1e-3, drop=1.0):
    L = [layers.Input(shape=(input_dim,)), layers.Dense(2048, activation="relu"), layers.Dropout(0.3 * drop)]
    if deep:
        L += [layers.Dense(256, activation="relu"), layers.Dropout(0.5 * drop),
              layers.Dense(128, activation="relu"), layers.Dropout(0.5 * drop)]
    L += [layers.Dense(K, activation="softmax")]
    m = kmodels.Sequential(L)
    m.compile(optimizer=tf.keras.optimizers.Adam(lr), loss="categorical_crossentropy", metrics=["accuracy"], jit_compile=False)  # no XLA: its compiled graphs are never freed and exhaust Colab RAM over hundreds of fits
    return m

def build_bilstm(vocab, deep=True, lr=1e-3, drop=1.0):
    inp = layers.Input(shape=(120,))
    x = layers.Embedding(vocab, 128)(inp)
    if deep:
        x = layers.SpatialDropout1D(0.2 * drop)(x)
        x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
        x = layers.Concatenate()([layers.GlobalMaxPooling1D()(x), layers.GlobalAveragePooling1D()(x)])
        x = layers.Dropout(0.4 * drop)(x); x = layers.Dense(256, activation="relu")(x); x = layers.Dropout(0.4 * drop)(x)
    else:
        x = layers.Bidirectional(layers.LSTM(64))(x); x = layers.Dropout(0.4 * drop)(x)
    out = layers.Dense(K, activation="softmax")(x)
    m = kmodels.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(lr), loss="categorical_crossentropy", metrics=["accuracy"], jit_compile=False)  # no XLA: its compiled graphs are never freed and exhaust Colab RAM over hundreds of fits
    return m

KERAS_SPECS = {   # name -> (input kind, builder kwargs)
    "ANN":                 ("tfidf",  dict(kind="dense", deep=False)),
    "NFReqANN-TFIDF":      ("tfidf",  dict(kind="dense", deep=True)),
    "NFReqANN-DistilBERT": ("bert",   dict(kind="dense", deep=True)),
    "NFReqANN-Hybrid":     ("hybrid", dict(kind="dense", deep=True)),
    "BiLSTM":              ("seq",    dict(kind="lstm", deep=False)),
    "NFReqBiLSTM":         ("seq",    dict(kind="lstm", deep=True)),
}

def make_inputs(inp, sp):
    """Builds train/val/test matrices for one split. TF-IDF and the sequence tokenizer are fitted on the inner-train part only."""
    tr, va, te = sp["train"], sp["val"], sp["test"]
    ctx = {}
    if inp in ("tfidf", "hybrid"):
        tfv = TfidfVectorizer(max_features=MAX_FEATURES, min_df=1, smooth_idf=True, norm="l2")
        Xtr = tfv.fit_transform([CLEAN[i] for i in tr]).toarray().astype(np.float32)
        Xva = tfv.transform([CLEAN[i] for i in va]).toarray().astype(np.float32)
        Xte = tfv.transform([CLEAN[i] for i in te]).toarray().astype(np.float32)
        ctx["tfidf"] = tfv; ctx["n_tfidf"] = Xtr.shape[1]
        if inp == "hybrid":
            Xtr, Xva, Xte = [np.hstack([a, BERT_EMB[idx]]).astype(np.float32) for a, idx in ((Xtr, tr), (Xva, va), (Xte, te))]
        return Xtr, Xva, Xte, ctx
    if inp == "bert":
        return BERT_EMB[tr], BERT_EMB[va], BERT_EMB[te], ctx
    tk = Tokenizer(num_words=30000, oov_token="<OOV>"); tk.fit_on_texts([CLEAN[i] for i in tr])
    f = lambda idx: pad_sequences(tk.texts_to_sequences([CLEAN[i] for i in idx]), maxlen=120, padding="post", truncating="post")
    ctx["vocab"] = min(30000, len(tk.word_index) + 1)
    return f(tr), f(va), f(te), ctx

def fit_keras(name, Xtr, ytr, Xva, yva, ctx, seed, lr=1e-3, drop=1.0, class_weight=None):
    set_seed(seed)
    spec = KERAS_SPECS[name][1]
    m = build_dense(Xtr.shape[1], spec["deep"], lr, drop) if spec["kind"] == "dense" else build_bilstm(ctx["vocab"], spec["deep"], lr, drop)
    cb = [callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, min_lr=1e-6, verbose=0),
          callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)]
    Ytr, Yva = np.eye(K)[ytr], np.eye(K)[yva]
    t0 = time.time()
    h = m.fit(Xtr, Ytr, epochs=KERAS_EPOCHS, batch_size=KERAS_BATCH, validation_data=(Xva, Yva), callbacks=cb, verbose=0,
              class_weight=class_weight)
    return m, time.time() - t0, len(h.history["loss"])

# ---------------------------------------------------------------- fine-tuned transformers
class _DS(torch.utils.data.Dataset):
    def __init__(self, texts, labels): self.t, self.l = texts, labels
    def __len__(self): return len(self.t)
    def __getitem__(self, i): return self.t[i], int(self.l[i])

def _collate(tok):
    def f(b):
        enc = tok([x[0] for x in b], truncation=True, max_length=TR_MAX_LEN, padding=True, return_tensors="pt")
        enc = {k: v for k, v in enc.items() if k in ("input_ids", "attention_mask")}
        enc["labels"] = torch.tensor([x[1] for x in b]); return enc
    return f

def tr_predict(model, tok, texts, device, batch=32):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(texts), batch):
            enc = tok(texts[i:i + batch], truncation=True, max_length=TR_MAX_LEN, padding=True, return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items() if k in ("input_ids", "attention_mask")}
            out.append(torch.softmax(model(**enc).logits, -1).cpu().numpy())
    return np.vstack(out)

def fit_transformer(name, tr_idx, va_idx, seed, lr):
    set_seed(seed)
    tok, model = load_tok_model(name, "cls"); model.to(DEVICE)
    dl = torch.utils.data.DataLoader(_DS([SENT[i] for i in tr_idx], Y_ALL[tr_idx]), batch_size=TR_BATCH, shuffle=True, collate_fn=_collate(tok))
    dv = torch.utils.data.DataLoader(_DS([SENT[i] for i in va_idx], Y_ALL[va_idx]), batch_size=32, shuffle=False, collate_fn=_collate(tok))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=TR_WD)
    total = len(dl) * TR_MAX_EPOCHS
    sch = get_linear_schedule_with_warmup(opt, int(TR_WARMUP * total), total)
    best, best_state, bad, ep_run = 1e9, None, 0, 0
    t0 = time.time()
    for ep in range(TR_MAX_EPOCHS):
        model.train()
        for b in dl:
            b = {k: v.to(DEVICE) for k, v in b.items()}
            loss = model(**b).loss; opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sch.step()
        model.eval(); vl, n = 0.0, 0
        with torch.no_grad():
            for b in dv:
                b = {k: v.to(DEVICE) for k, v in b.items()}; vl += model(**b).loss.item() * len(b["labels"]); n += len(b["labels"])
        vl /= n; ep_run = ep + 1
        if vl < best - 1e-4:
            best, bad = vl, 0; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= TR_PATIENCE: break
    if best_state is not None: model.load_state_dict(best_state)
    return tok, model, time.time() - t0, ep_run

# ## Runner with per-fit checkpoints, hyper-parameter selection and block-level attribution

RUN_DIR = os.path.join(OUT_DIR, "runs"); ATTR_DIR = os.path.join(OUT_DIR, "attribution")
os.makedirs(RUN_DIR, exist_ok=True); os.makedirs(ATTR_DIR, exist_ok=True)
LR_FILE = os.path.join(OUT_DIR, "transformer_lr_selection.json")

def _ckpt(protocol, name, fold): return os.path.join(RUN_DIR, protocol, name, f"{fold}.json")

def select_transformer_lr(name):
    """Learning rate chosen once, on the inner validation split of seed 34 only (test data never used)."""
    sel = json.load(open(LR_FILE)) if os.path.exists(LR_FILE) else {}
    if name in sel: return sel[name]["best_lr"]
    sp = next(splits_A()); scores = {}
    for lr in TR_LR_GRID:
        tok, m, _, ep = fit_transformer(name, sp["train"], sp["val"], sp["seed"], lr)
        pv = tr_predict(m, tok, [SENT[i] for i in sp["val"]], DEVICE).argmax(1)
        scores[str(lr)] = float(macro_f1(Y_ALL[sp["val"]], pv)); print(f"  {name} lr={lr}: val macro-F1={scores[str(lr)]:.4f} ({ep} ep)")
        del m; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    best = float(max(scores, key=scores.get)); sel[name] = dict(best_lr=best, val_macro_f1=scores)
    json.dump(sel, open(LR_FILE, "w"), indent=1); return best

def _linear(kind, C, cw, seed):
    return LinearSVC(C=C, class_weight=cw, random_state=seed, max_iter=5000) if kind == "SVM" else \
           LogisticRegression(C=C, class_weight=cw, random_state=seed, max_iter=2000)

def _tfidf_fit(idx_fit, *idx_other):
    tfv = TfidfVectorizer(max_features=MAX_FEATURES, min_df=1, smooth_idf=True, norm="l2")
    out = [tfv.fit_transform([CLEAN[i] for i in idx_fit])] + [tfv.transform([CLEAN[i] for i in o]) for o in idx_other]
    return out

def block_attribution(m, Xtr, Xte, yte, n_tfidf, tfv, seed):
    """(i) exact two-player Shapley values of the TF-IDF and DistilBERT blocks (absent block = training mean),
       (ii) block-removal performance, (iii) expected-gradients attributions for all 5,768 inputs."""
    P = lambda X: m.predict(X, batch_size=256, verbose=0)
    mT, mB = Xtr[:, :n_tfidf].mean(0), Xtr[:, n_tfidf:].mean(0)
    noT, noB, none = Xte.copy(), Xte.copy(), Xte.copy()
    noT[:, :n_tfidf] = mT; noB[:, n_tfidf:] = mB; none[:, :n_tfidf] = mT; none[:, n_tfidf:] = mB
    pf, pT, pB, p0 = P(Xte), P(noT), P(noB), P(none)
    c = pf.argmax(1); g = lambda p: p[np.arange(len(c)), c]
    phiT = 0.5 * ((g(pf) - g(pT)) + (g(pB) - g(p0)))
    phiB = 0.5 * ((g(pf) - g(pB)) + (g(pT) - g(p0)))
    share_T = np.abs(phiT) / (np.abs(phiT) + np.abs(phiB) + 1e-12)
    res = dict(seed=seed,
               shapley_tfidf_mean=float(phiT.mean()), shapley_bert_mean=float(phiB.mean()),
               shapley_share_tfidf=float(share_T.mean()),
               shapley_share_tfidf_by_true_class={CATS[k]: float(share_T[yte == k].mean()) for k in range(K)},
               acc_full=float(accuracy_score(yte, pf.argmax(1))), f1_full=float(macro_f1(yte, pf.argmax(1))),
               acc_tfidf_removed=float(accuracy_score(yte, pT.argmax(1))), f1_tfidf_removed=float(macro_f1(yte, pT.argmax(1))),
               acc_bert_removed=float(accuracy_score(yte, pB.argmax(1))), f1_bert_removed=float(macro_f1(yte, pB.argmax(1))),
               f1_by_class_full=list(f1_score(yte, pf.argmax(1), average=None, labels=list(range(K)), zero_division=0)),
               f1_by_class_tfidf_removed=list(f1_score(yte, pT.argmax(1), average=None, labels=list(range(K)), zero_division=0)),
               f1_by_class_bert_removed=list(f1_score(yte, pB.argmax(1), average=None, labels=list(range(K)), zero_division=0)))
    # expected gradients (Erion et al. 2021) w.r.t. the predicted-class probability
    rng = np.random.default_rng(seed); bg = Xtr[rng.choice(len(Xtr), size=min(100, len(Xtr)), replace=False)]
    S, D = 64, Xte.shape[1]; A = np.zeros_like(Xte)
    for i in range(0, len(Xte), 16):
        xb, cb = Xte[i:i + 16], c[i:i + 16]; B = len(xb)
        b = bg[rng.integers(0, len(bg), size=(B, S))]
        a = rng.uniform(size=(B, S, 1)).astype(np.float32)
        pts = tf.convert_to_tensor((b + a * (xb[:, None, :] - b)).reshape(-1, D))
        oh = tf.convert_to_tensor(np.repeat(np.eye(K, dtype=np.float32)[cb], S, axis=0))
        with tf.GradientTape() as tape:
            tape.watch(pts); out = tf.reduce_sum(m(pts, training=False) * oh, axis=1)
        gr = tape.gradient(out, pts).numpy().reshape(B, S, D)
        A[i:i + 16] = ((xb[:, None, :] - b) * gr).mean(1)
    absT, absB = np.abs(A[:, :n_tfidf]).sum(1), np.abs(A[:, n_tfidf:]).sum(1)
    res["eg_share_tfidf"] = float((absT / (absT + absB + 1e-12)).mean())
    res["eg_signed_tfidf_mean"] = float(A[:, :n_tfidf].sum(1).mean()); res["eg_signed_bert_mean"] = float(A[:, n_tfidf:].sum(1).mean())
    res["eg_completeness_gap"] = float(np.mean(np.abs(A.sum(1) - (g(pf) - P(bg).mean(0)[c]))))
    vocab = np.array(tfv.get_feature_names_out()); imp = np.abs(A[:, :n_tfidf]).mean(0)
    top = np.argsort(-imp)[:50]
    res["top_tfidf_tokens"] = [(vocab[j], float(imp[j])) for j in top]
    res["mean_abs_eg_per_dim_tfidf"] = float(np.abs(A[:, :n_tfidf]).mean()); res["mean_abs_eg_per_dim_bert"] = float(np.abs(A[:, n_tfidf:]).mean())
    return res

def run_one(protocol, name, sp):
    path = _ckpt(protocol, name, sp["fold"])
    if os.path.exists(path): return json.load(open(path))
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tr, va, te, otr, seed = sp["train"], sp["val"], sp["test"], sp["outer_train"], sp["seed"]
    ytr, yva, yte = Y_ALL[tr], Y_ALL[va], Y_ALL[te]
    info, t_inf = {}, None
    base = name.replace("-tuned", "").replace("-cw", "")
    if base in ("SVM", "LogReg"):
        if name.endswith("-tuned"):
            Xtr, Xva = _tfidf_fit(tr, va); best = (-1, None)
            for C in LINEAR_C_GRID:
                for cw in (None, "balanced"):
                    s = macro_f1(yva, _linear(base, C, cw, seed).fit(Xtr, ytr).predict(Xva))
                    if s > best[0]: best = (s, (C, cw))
            C, cw = best[1]; info = dict(C=C, class_weight=cw, val_macro_f1=best[0])
        else:
            C, cw = 1.0, None
        Xo, Xte = _tfidf_fit(otr, te)          # refit on the full outer-training partition (as in the manuscript)
        t0 = time.time(); clf = _linear(base, C, cw, seed).fit(Xo, Y_ALL[otr]); t_tr = time.time() - t0
        t0 = time.time(); pred = clf.predict(Xte); t_inf = (time.time() - t0) / len(te)
        proba = None; epochs = None
    elif base in KERAS_SPECS:
        inp = KERAS_SPECS[base][0]
        Xtr, Xva, Xte, ctx = make_inputs(inp, sp)
        cw = None
        if name.endswith("-cw"):
            w = compute_class_weight("balanced", classes=np.arange(K), y=ytr); cw = {i: float(w[i]) for i in range(K)}
            info["class_weight"] = cw
        hp = dict(lr=1e-3, drop=1.0)
        if name.endswith("-tuned"):
            best = (-1, None)
            for g in KERAS_GRID:
                mm, _, _ = fit_keras(base, Xtr, ytr, Xva, yva, ctx, seed, **g)
                s = macro_f1(yva, mm.predict(Xva, verbose=0).argmax(1)); best = max(best, (s, json.dumps(g)))
                del mm; tf.keras.backend.clear_session(); gc.collect()
            hp = json.loads(best[1]); info = dict(selected=hp, val_macro_f1=best[0])
        m, t_tr, epochs = fit_keras(base, Xtr, ytr, Xva, yva, ctx, seed, class_weight=cw, **hp)
        m.predict(Xte[:8], verbose=0); t0 = time.time(); proba = m.predict(Xte, verbose=0); t_inf = (time.time() - t0) / len(te)
        pred = proba.argmax(1)
        if protocol == "A" and name == "NFReqANN-Hybrid" and RUN_ATTRIBUTION:
            ap = os.path.join(ATTR_DIR, f"{sp['fold']}.json")
            if not os.path.exists(ap):
                json.dump(block_attribution(m, Xtr, Xte, yte, ctx["n_tfidf"], ctx["tfidf"], seed), open(ap, "w"))
        del m; tf.keras.backend.clear_session(); gc.collect()
    elif name in ("FT-DistilBERT", "FT-RoBERTa"):
        lr = select_transformer_lr(DISTILBERT if name == "FT-DistilBERT" else ROBERTA)
        tok, m, t_tr, epochs = fit_transformer(DISTILBERT if name == "FT-DistilBERT" else ROBERTA, tr, va, seed, lr)
        texts = [SENT[i] for i in te]; tr_predict(m, tok, texts[:8], DEVICE)
        t0 = time.time(); proba = tr_predict(m, tok, texts, DEVICE); t_inf = (time.time() - t0) / len(te)
        pred = proba.argmax(1); info = dict(lr=lr)
        del m; torch.cuda.empty_cache() if torch.cuda.is_available() else None
    else:
        raise ValueError(name)
    gc.collect()
    rec = dict(protocol=protocol, model=name, fold=sp["fold"], seed=int(seed), test_idx=[int(i) for i in te],
               y_true=[int(v) for v in yte], y_pred=[int(v) for v in pred],
               proba=None if proba is None else np.round(proba, 5).tolist(),
               n_train=int(len(tr)), n_val=int(len(va)), n_outer_train=int(len(otr)), n_test=int(len(te)),
               accuracy=float(accuracy_score(yte, pred)), f1_macro=float(macro_f1(yte, pred)),
               train_time_sec=float(t_tr), infer_sec_per_item=t_inf, epochs=epochs, info=info)
    json.dump(rec, open(path, "w"))
    print(f"  [{protocol}] {name:24s} {sp['fold']:8s} acc={rec['accuracy']:.4f} F1={rec['f1_macro']:.4f} ({t_tr:.1f}s)")
    return rec

def run_protocol(protocol, gen, names):
    for name in names:
        print(f"== Protocol {protocol}: {name}")
        for sp in gen():
            run_one(protocol, name, sp)

BASE_MODELS = ["SVM", "LogReg", "SVM-tuned", "LogReg-tuned", "ANN", "NFReqANN-TFIDF", "NFReqANN-DistilBERT",
               "NFReqANN-Hybrid", "BiLSTM", "NFReqBiLSTM", "FT-DistilBERT", "FT-RoBERTa"]
EXTRA_A = ["NFReqANN-Hybrid-cw"] + [f"{n}-tuned" for n in KERAS_SPECS]

# ## Run all protocols (resumable)

with open(os.path.join(OUT_DIR, f"environment_phase{PHASE}.json"), "w") as f:
    json.dump(dict(python=platform.python_version(), torch=torch.__version__, tensorflow=tf.__version__ if USE_TF else None, phase=PHASE,
                   transformers=__import__("transformers").__version__, sklearn=__import__("sklearn").__version__,
                   gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                   cpu_count=os.cpu_count(), local_test=LOCAL_TEST), f, indent=1)

if STAGE == "run":
    run_protocol(JOB_PROTOCOL, {"A": splits_A, "B": splits_B, "C": splits_C}[JOB_PROTOCOL], JOB_MODELS)

# ## End-to-end computational cost on GPU and CPU (embedding extraction included)

def _median_single(fn, items, n=50):
    ts = []
    for x in items[:n]:
        t0 = time.perf_counter(); fn([x]); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

TIMING_FILE = os.path.join(OUT_DIR, f"timing_end_to_end_phase{PHASE}.json")
if STAGE == "timing" and not os.path.exists(TIMING_FILE):
    sp = next(splits_A()); tr, va, te = sp["train"], sp["val"], sp["test"]
    fit_idx = np.r_[tr, va]; te_txt = [SENT[i] for i in te]
    runsA = lambda n: [json.load(open(f)) for f in glob.glob(os.path.join(RUN_DIR, "A", n, "*.json"))]
    rows = []
    for dev in (["cuda", "cpu"] if torch.cuda.is_available() else ["cpu"]):
        tdev, tfdev = torch.device(dev), ("/GPU:0" if dev == "cuda" else "/CPU:0")
        if dev == "cpu": torch.set_num_threads(os.cpu_count())
        if PHASE == 1:
            # ---------------- NFReqANN-Hybrid: everything from raw text
            tok, enc = load_tok_model(DISTILBERT, "encoder"); enc.to(tdev)
            embed_texts(SENT[:16], tok, enc, tdev)                                   # warm-up
            t0 = time.perf_counter()
            tfv = TfidfVectorizer(max_features=MAX_FEATURES, min_df=1, smooth_idf=True, norm="l2")
            Xt = tfv.fit_transform([clean_text(SENT[i]) for i in tr]).toarray().astype(np.float32)
            Xv = tfv.transform([clean_text(SENT[i]) for i in va]).toarray().astype(np.float32)
            t_tfidf = time.perf_counter() - t0
            t0 = time.perf_counter(); Et = embed_texts([SENT[i] for i in tr], tok, enc, tdev); Ev = embed_texts([SENT[i] for i in va], tok, enc, tdev)
            t_embed = time.perf_counter() - t0
            with tf.device(tfdev):
                m, t_fit, _ = fit_keras("NFReqANN-Hybrid", np.hstack([Xt, Et]), Y_ALL[tr], np.hstack([Xv, Ev]), Y_ALL[va], {}, 34)
                def hyb_predict(texts):
                    X = tfv.transform([clean_text(t) for t in texts]).toarray().astype(np.float32)
                    # direct call instead of m.predict(): avoids Keras' fixed per-call overhead (~50-70 ms) that would distort single-item latency
                    return m(tf.convert_to_tensor(np.hstack([X, embed_texts(texts, tok, enc, tdev)])), training=False).numpy()
                hyb_predict(te_txt[:8]); t0 = time.perf_counter(); hyb_predict(te_txt); t_inf = (time.perf_counter() - t0) / len(te)
                t_single = _median_single(hyb_predict, te_txt)
            n_ann = int(m.count_params()); n_enc = int(sum(p.numel() for p in enc.parameters()))
            rows.append(dict(model="NFReqANN-Hybrid", device=dev, train_tfidf_sec=t_tfidf, train_embedding_sec=t_embed, train_classifier_sec=t_fit,
                             train_total_sec=t_tfidf + t_embed + t_fit, train_total_is_estimate=False,
                             infer_batch_ms_per_item=1000 * t_inf, infer_single_ms=1000 * t_single,
                             trainable_params=n_ann, frozen_params=n_enc))
            del m, enc; tf.keras.backend.clear_session(); gc.collect()
        if PHASE == 2:
            # ---------------- fine-tuned transformers (inference cost does not depend on the weights' values)
            for label, hf in (("FT-DistilBERT", DISTILBERT), ("FT-RoBERTa", ROBERTA)):
                tok2, m2 = load_tok_model(hf, "cls"); m2.to(tdev)
                pf = lambda texts: tr_predict(m2, tok2, texts, tdev)
                pf(te_txt[:8]); t0 = time.perf_counter(); pf(te_txt); t_inf = (time.perf_counter() - t0) / len(te)
                t_single = _median_single(pf, te_txt)
                R = runsA(label)
                mean_ep = float(np.mean([r["epochs"] for r in R])) if R else float(TR_MAX_EPOCHS)
                if dev == "cuda" and R:
                    t_train, est = float(np.mean([r["train_time_sec"] for r in R])), False
                else:   # CPU: time 10 optimisation steps and extrapolate to the mean number of epochs used in Protocol A
                    dl = torch.utils.data.DataLoader(_DS([SENT[i] for i in tr], Y_ALL[tr]), batch_size=TR_BATCH, shuffle=True, collate_fn=_collate(tok2))
                    opt = torch.optim.AdamW(m2.parameters(), lr=2e-5); m2.train(); it = iter(dl); n_steps = 0
                    b = {k: v.to(tdev) for k, v in next(it).items()}; m2(**b).loss.backward(); opt.step(); opt.zero_grad()   # warm-up
                    t0 = time.perf_counter()
                    for b in it:
                        b = {k: v.to(tdev) for k, v in b.items()}; m2(**b).loss.backward(); opt.step(); opt.zero_grad(); n_steps += 1
                        if n_steps == 10: break
                    per_step = (time.perf_counter() - t0) / n_steps
                    t_train, est = per_step * len(dl) * mean_ep * 1.15, True     # +15% for per-epoch validation passes
                rows.append(dict(model=label, device=dev, train_total_sec=t_train, train_total_is_estimate=est,
                                 infer_batch_ms_per_item=1000 * t_inf, infer_single_ms=1000 * t_single,
                                 trainable_params=int(sum(p.numel() for p in m2.parameters())), frozen_params=0, mean_epochs=mean_ep))
                del m2; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    if PHASE == 1:
        # ---------------- tuned linear SVM (CPU only)
        t0 = time.perf_counter()
        tfv = TfidfVectorizer(max_features=MAX_FEATURES, min_df=1, smooth_idf=True, norm="l2")
        Xo = tfv.fit_transform([clean_text(SENT[i]) for i in sp["outer_train"]])
        clf = LinearSVC(C=1.0, max_iter=5000).fit(Xo, Y_ALL[sp["outer_train"]]); t_svm = time.perf_counter() - t0
        t0 = time.perf_counter(); clf.predict(tfv.transform([clean_text(t) for t in te_txt])); t_svm_inf = (time.perf_counter() - t0) / len(te)
        rows.append(dict(model="SVM", device="cpu", train_total_sec=t_svm, train_total_is_estimate=False, infer_batch_ms_per_item=1000 * t_svm_inf))
    json.dump(rows, open(TIMING_FILE, "w"), indent=1)
_tfiles = sorted(glob.glob(os.path.join(OUT_DIR, "timing_end_to_end_phase*.json")))
if _tfiles and STAGE in ("timing", "summary"):
    print(pd.DataFrame([r for f in _tfiles for r in json.load(open(f))]).round(3).to_string(index=False))

# ## Summary tables and statistics
# * Protocols A and B: paired comparisons against NFReqANN-Hybrid with the **corrected resampled t-test** (Nadeau & Bengio 2003;
#   for repeated k-fold this is the Bouckaert & Frank 2004 correction), on accuracy **and** macro-F1, Holm-adjusted across comparators.
#   The uncorrected paired t-test and the Wilcoxon signed-rank test are reported for reference only.
# * Protocol C (leave-one-project-out): metrics are computed on the pooled out-of-project predictions; differences are tested with a
#   project-level cluster bootstrap (5,000 resamples of projects).

from scipy import stats
SUM_DIR = os.path.join(OUT_DIR, "summary"); os.makedirs(SUM_DIR, exist_ok=True)
REF = "NFReqANN-Hybrid"

def load_runs(protocol):
    R = [json.load(open(f)) for f in glob.glob(os.path.join(RUN_DIR, protocol, "*", "*.json"))]
    return pd.DataFrame(R)

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for k, i in enumerate(o):
        run = max(run, (len(p) - k) * p[i]); adj[i] = min(1.0, run)
    return adj

def per_fold_metrics(R):
    rows = []
    for _, r in R.iterrows():
        yt, yp = np.array(r.y_true), np.array(r.y_pred)
        P, Rc, F, _ = precision_recall_fscore_support(yt, yp, labels=list(range(K)), zero_division=0)
        d = dict(model=r.model, fold=r.fold, accuracy=accuracy_score(yt, yp), precision_macro=P.mean(), recall_macro=Rc.mean(),
                 f1_macro=F.mean(), n_test=r.n_test, n_outer_train=r.n_outer_train, train_time_sec=r.train_time_sec,
                 infer_ms_per_item=None if r.infer_sec_per_item is None else 1000 * r.infer_sec_per_item)
        for k in range(K): d[f"f1_{CATS[k]}"] = F[k]; d[f"recall_{CATS[k]}"] = Rc[k]; d[f"precision_{CATS[k]}"] = P[k]
        rows.append(d)
    return pd.DataFrame(rows)

def corrected_tests(M, ref=REF):
    out = []
    for metric in ("accuracy", "f1_macro"):
        piv = M.pivot(index="fold", columns="model", values=metric).dropna(axis=1)
        if ref not in piv: continue
        ratio = float((M.n_test / M.n_outer_train).mean())
        res = []
        for c in [c for c in piv.columns if c != ref]:
            d = (piv[ref] - piv[c]).values; J = len(d); mu, sd = d.mean(), d.std(ddof=1)
            se_c = np.sqrt(1 / J + ratio) * sd if sd > 0 else np.nan
            t_c = mu / se_c if sd > 0 else np.nan
            p_c = 2 * stats.t.sf(abs(t_c), J - 1) if sd > 0 else 1.0
            p_t = 2 * stats.t.sf(abs(mu / (sd / np.sqrt(J))), J - 1) if sd > 0 else 1.0
            try: p_w = stats.wilcoxon(d).pvalue
            except ValueError: p_w = 1.0
            tc = stats.t.ppf(0.975, J - 1)
            res.append(dict(metric=metric, comparator=c, n_pairs=J, mean_diff=mu, ci95_low=mu - tc * se_c, ci95_high=mu + tc * se_c,
                            p_corrected=p_c, p_paired_t_uncorrected=p_t, p_wilcoxon=p_w, cohen_dz=mu / sd if sd > 0 else np.nan))
        res = pd.DataFrame(res); res["p_corrected_holm"] = holm(res.p_corrected); res["significant"] = res.p_corrected_holm < 0.05
        out.append(res)
    return pd.concat(out) if out else pd.DataFrame()

def summarize_AB(protocol):
    R = load_runs(protocol)
    if R.empty: return
    M = per_fold_metrics(R); M.to_csv(os.path.join(SUM_DIR, f"{protocol}_per_fold.csv"), index=False)
    cols = ["accuracy", "precision_macro", "recall_macro", "f1_macro"] + [f"f1_{c}" for c in CATS] + ["train_time_sec", "infer_ms_per_item"]
    agg = M.groupby("model")[cols].agg(["mean", "std"]); agg.columns = [f"{a}_{b}" for a, b in agg.columns]
    agg["n_folds"] = M.groupby("model").size(); agg = agg.sort_values("f1_macro_mean", ascending=False)
    agg.to_csv(os.path.join(SUM_DIR, f"{protocol}_aggregate.csv"))
    T = corrected_tests(M); T.to_csv(os.path.join(SUM_DIR, f"{protocol}_tests_vs_hybrid.csv"), index=False)
    print(f"\n===== Protocol {protocol} ====="); print(agg[[c for c in agg.columns if c.split('_')[0] in ('accuracy', 'f1', 'n')]].round(4).to_string())
    if not T.empty: print(T[["metric", "comparator", "mean_diff", "ci95_low", "ci95_high", "p_corrected", "p_corrected_holm", "significant"]].round(4).to_string(index=False))

def summarize_C(n_boot=5000):
    R = load_runs("C")
    if R.empty: return
    rows, pooled = [], {}
    proj_of = lambda f: int(f[1:])
    for name, g in R.groupby("model"):
        yt = np.concatenate(g.y_true.values); yp = np.concatenate(g.y_pred.values)
        P, Rc, F, S = precision_recall_fscore_support(yt, yp, labels=list(range(K)), zero_division=0)
        d = dict(model=name, n_projects=len(g), n_test=len(yt), accuracy=accuracy_score(yt, yp), precision_macro=P.mean(),
                 recall_macro=Rc.mean(), f1_macro=F.mean(), f1_macro_4cls_excl_maint=F[1:].mean())
        for k in range(K): d[f"f1_{CATS[k]}"] = F[k]; d[f"support_{CATS[k]}"] = int(S[k])
        rows.append(d); pooled[name] = {proj_of(f): (np.array(a), np.array(b)) for f, a, b in zip(g.fold, g.y_true, g.y_pred)}
        cm = confusion_matrix(yt, yp, labels=list(range(K)))
        pd.DataFrame(cm, index=CATS, columns=CATS).to_csv(os.path.join(SUM_DIR, f"C_confusion_{name}.csv"))
    A = pd.DataFrame(rows).sort_values("f1_macro", ascending=False); A.to_csv(os.path.join(SUM_DIR, "C_pooled.csv"), index=False)
    # project-cluster bootstrap of differences vs the hybrid
    rng = np.random.default_rng(0); res = []
    if REF in pooled:
        projs = sorted(pooled[REF])
        for c in [c for c in pooled if c != REF and set(pooled[c]) == set(projs)]:
            diffs = {"accuracy": [], "f1_macro": []}
            def met(name, sample):
                yt = np.concatenate([pooled[name][p][0] for p in sample]); yp = np.concatenate([pooled[name][p][1] for p in sample])
                return accuracy_score(yt, yp), macro_f1(yt, yp)
            obs = np.subtract(met(REF, projs), met(c, projs))
            for _ in range(n_boot):
                s = rng.choice(projs, size=len(projs), replace=True); a, b = met(REF, s), met(c, s)
                diffs["accuracy"].append(a[0] - b[0]); diffs["f1_macro"].append(a[1] - b[1])
            for i, mname in enumerate(("accuracy", "f1_macro")):
                dd = np.array(diffs[mname]); p = 2 * min((dd <= 0).mean(), (dd >= 0).mean())
                res.append(dict(metric=mname, comparator=c, diff=obs[i], ci95_low=np.percentile(dd, 2.5), ci95_high=np.percentile(dd, 97.5), p_boot=min(1, p)))
        res = pd.DataFrame(res)
        for mname in ("accuracy", "f1_macro"):
            msk = res.metric == mname; res.loc[msk, "p_boot_holm"] = holm(res.loc[msk, "p_boot"])
        res.to_csv(os.path.join(SUM_DIR, "C_bootstrap_vs_hybrid.csv"), index=False)
    print("\n===== Protocol C (leave-one-project-out, pooled) ====="); print(A.round(4).to_string(index=False))
    if len(res): print(pd.DataFrame(res).round(4).to_string(index=False))

def summarize_attribution():
    F = sorted(glob.glob(os.path.join(ATTR_DIR, "*.json")))
    if not F: return
    A = [json.load(open(f)) for f in F]
    scal = pd.DataFrame([{k: v for k, v in a.items() if not isinstance(v, (list, dict))} for a in A])
    scal.to_csv(os.path.join(SUM_DIR, "attribution_per_seed.csv"), index=False)
    print("\n===== Block-level attribution (NFReqANN-Hybrid, Protocol A) ====="); print(scal.describe().loc[["mean", "std"]].T.round(4).to_string())
    byc = pd.DataFrame([a["shapley_share_tfidf_by_true_class"] for a in A]); byc.to_csv(os.path.join(SUM_DIR, "attribution_share_by_class.csv"), index=False)
    pc = pd.DataFrame({f"{w}_{CATS[k]}": [a[f"f1_by_class_{w}"][k] for a in A] for w in ("full", "tfidf_removed", "bert_removed") for k in range(K)})
    pc.to_csv(os.path.join(SUM_DIR, "attribution_f1_by_class.csv"), index=False)
    tok = {}
    for a in A:
        for rank, (t, v) in enumerate(a["top_tfidf_tokens"][:30]):
            tok.setdefault(t, []).append(v)
    tokdf = pd.DataFrame([dict(token=t, n_seeds_in_top30=len(v), mean_abs_eg=np.mean(v)) for t, v in tok.items()]).sort_values(["n_seeds_in_top30", "mean_abs_eg"], ascending=False)
    tokdf.to_csv(os.path.join(SUM_DIR, "attribution_top_tokens.csv"), index=False); print(tokdf.head(30).to_string(index=False))

if STAGE == "summary":
    # export the exact index files of every split (released with the paper)
    _rows = []
    for _p, _gen in (("A", splits_A), ("B", splits_B), ("C", splits_C)):
        for _sp in _gen():
            for _role in ("train", "val", "test"):
                _rows += [dict(protocol=_p, fold=_sp["fold"], role=_role, row_id=int(i)) for i in _sp[_role]]
    pd.DataFrame(_rows).to_csv(os.path.join(OUT_DIR, "split_indices.csv"), index=False)
    for p in ("A", "B"): summarize_AB(p)
    summarize_C(); summarize_attribution()
    zip_path = shutil.make_archive(os.path.join(BASE, "nfr_revision_results"), "zip", OUT_DIR)
    print("\nResults archive:", zip_path)


### 2. Run everything

In [ ]:
# ============================== SETTINGS ==============================
USE_DRIVE = False      # True = keep results in Google Drive (survives a full Colab disconnect)
# ======================================================================
import os, sys, subprocess, shutil, json, glob, time
IN_COLAB = "google.colab" in sys.modules
LOCAL_TEST = os.environ.get("NFR_LOCAL_TEST", "0") == "1"
HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else "/content"
CORE = os.path.join(HERE, "nfr_core.py")

if IN_COLAB and USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive"); BASE = "/content/drive/MyDrive/nfr_revision"
elif IN_COLAB:
    BASE = "/content/nfr_revision"
else:
    BASE = os.environ.get("NFR_BASE", os.path.abspath("./nfr_revision"))
os.makedirs(os.path.join(BASE, "logs"), exist_ok=True)
DATA = os.path.join(BASE, "nfr_five_provenance.csv")
if not os.path.exists(DATA):
    if IN_COLAB:
        from google.colab import files
        print("Upload nfr_five_provenance.csv"); up = files.upload(); shutil.move(list(up)[0], DATA)
    else:
        shutil.copy("nfr_five_provenance.csv", DATA)
if IN_COLAB:   # triton (unused) clashes with TensorFlow's LLVM; memory_profiler not needed
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "triton"], check=False)

# ------------------------------ job list ------------------------------
LINEAR = "SVM,LogReg,SVM-tuned,LogReg-tuned"
KERAS = ["ANN", "NFReqANN-TFIDF", "NFReqANN-DistilBERT", "NFReqANN-Hybrid", "BiLSTM", "NFReqBiLSTM"]
FT = ["FT-DistilBERT", "FT-RoBERTa"]
N_FOLDS = {"A": 2, "B": 5, "C": 3} if LOCAL_TEST else {"A": 10, "B": 10, "C": 15}
JOBS = []   # (phase, stage, protocol, models)
for p in "ABC":
    JOBS.append((1, "run", p, LINEAR))
    JOBS += [(1, "run", p, m) for m in KERAS]
JOBS.append((1, "run", "A", "NFReqANN-Hybrid-cw"))
for p in "ABC":
    JOBS += [(2, "run", p, m) for m in FT]
JOBS += [(1, "run", "A", f"{m}-tuned") for m in KERAS]
JOBS += [(1, "timing", "", ""), (2, "timing", "", ""), (2, "summary", "", "")]

def done(job):
    ph, st, p, models = job
    out = os.path.join(BASE, "results")
    if st == "timing": return os.path.exists(os.path.join(out, f"timing_end_to_end_phase{ph}.json"))
    if st == "summary": return False
    return all(len(glob.glob(os.path.join(out, "runs", p, m, "*.json"))) >= N_FOLDS[p] for m in models.split(","))

KEEP = ("== Protocol", "[A]", "[B]", "[C]", "lr=", "Traceback", "Error", "error", "=====", "Results archive", "model device")
def run_job(i, job):
    ph, st, p, models = job
    env = dict(os.environ, NFR_PHASE=str(ph), NFR_STAGE=st, NFR_PROTOCOL=p, NFR_MODELS=models, NFR_BASE=BASE,
               TF_CPP_MIN_LOG_LEVEL="3", PYTHONUNBUFFERED="1")
    log = os.path.join(BASE, "logs", f"{i:02d}_{st}_{p}_{models.replace(',', '+')[:40]}.log")
    with open(log, "w") as lf:
        proc = subprocess.Popen([sys.executable, CORE], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=BASE)
        for line in proc.stdout:
            lf.write(line)
            if st == "summary" or any(k in line for k in KEEP): print("   " + line.rstrip())
        proc.wait()
    return proc.returncode, log

t_start = time.time()
for i, job in enumerate(JOBS, 1):
    label = f"[{i}/{len(JOBS)}] {job[1]} {job[2]} {job[3]}".strip()
    if done(job):
        print(f"{label}: already done"); continue
    for attempt in (1, 2, 3):
        print(f"{label}" + (f" (retry {attempt - 1})" if attempt > 1 else ""), flush=True)
        rc, log = run_job(i, job)
        if rc == 0 and (done(job) or job[1] == "summary"): break
        print(f"   job ended with exit code {rc}; resuming from its checkpoints ...")
    else:
        print("\n".join(open(log).read().splitlines()[-40:]))
        raise SystemExit(f"STOPPED: '{label}' failed 3 times. Send the lines above (full log: {log}).")
print(f"\nALL DONE in {(time.time() - t_start) / 60:.0f} min.")
ZIP = os.path.join(BASE, "nfr_revision_results.zip")
if IN_COLAB and os.path.exists(ZIP):
    from google.colab import files; files.download(ZIP)
